In [ ]:
# 1. Uninstall everything first to remove conflicts
!pip uninstall -y torch torchvision torchaudio peft transformers accelerate bitsandbytes trl

# 2. Reinstall compatible versions (PyTorch 2.1+ is recommended for Llama 2)
# We install torch first to ensure the base is correct
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3. Install the specific fine-tuning libraries
!pip install -U peft transformers accelerate bitsandbytes trl datasets scipy

In [ ]:
from huggingface_hub import login
import numpy
import os
# Replace with your actual token starting with 'hf_...'
login(token = os.getenv("HF_TOKEN"))

In [2]:
import torch
import gc
import os
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from trl import SFTTrainer, SFTConfig

# --- 1. Memory Cleanup ---
# Restarting the kernel is the best way, but this helps:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

# Configuration
model_id = "microsoft/Llama2-7b-WhoIsHarryPotter"
new_model_name = "llama2-hp-task1-adapter"
dataset_file = "/kaggle/input/philospher-stone-prashans/harry_potter_train_instruct.json"

# --- 2. Load Dataset ---
dataset = load_dataset("json", data_files=dataset_file, split="train")

def format_instruction(sample):
    return f"<s>[INST] {sample['instruction']} [/INST] {sample['output']} </s>"

# --- 3. Load Tokenizer & Model ---
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# We keep quantization efficient
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, 
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16 
)

# Prepare model (Cast Layernorms to float32)
base_model.gradient_checkpointing_enable()
base_model = prepare_model_for_kbit_training(base_model)

# --- 4. LoRA Configuration ---
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

# --- 5. Training Configuration ---
sft_config = SFTConfig(
    output_dir="./results_task1",
    dataset_text_field="output",
    max_length=512,  
    packing=False,
    
    num_train_epochs=3,
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    
    # --- THE FIX IS HERE ---
    # We DISABLE both flags. 
    # The model will still compute in float16 because of bnb_config above,
    # but the problematic "GradScaler" will be turned off.
    fp16=False, 
    bf16=False,
    
    learning_rate=2e-4,
    weight_decay=0.001,
    logging_steps=25,
    save_strategy="epoch",
    report_to="none"
)

# --- 6. Trainer ---
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=format_instruction,
    processing_class=tokenizer,
    args=sft_config,
)

# --- 7. Train & Save ---
print("Starting training (Scaler Disabled)...")
trainer.train()

trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print("Task 1 Training Complete. Adapters saved.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training (Scaler Disabled)...


Step,Training Loss
25,2.352593
50,2.023983
75,1.936330
100,1.871024


Task 1 Training Complete. Adapters saved.


In [3]:
import torch
from tqdm import tqdm
import json
import random

# --- Configuration ---
SAMPLES_FOR_FISHER = 500  # How many examples to estimate importance
REPLAY_BUFFER_SIZE = 200  # How many examples to save for later

# 1. Calculate Fisher Information Matrix (EWC)
# We calculate how important every trainable parameter (LoRA weight) is.
print(f"Calculating Fisher Matrix on {SAMPLES_FOR_FISHER} samples...")

model = trainer.model # Use the model currently in memory
model.eval() 

fisher_dict = {}
opt_param_dict = {}

# Initialize dicts
for name, param in model.named_parameters():
    if param.requires_grad:
        fisher_dict[name] = torch.zeros_like(param)
        opt_param_dict[name] = param.data.clone() # Save current "best" weights

# Shuffle dataset to get a random representative sample
eval_subset = dataset.shuffle(seed=42).select(range(min(len(dataset), SAMPLES_FOR_FISHER)))

# Loop to calculate gradients
for i, sample in enumerate(tqdm(eval_subset)):
    # Re-create the prompt format used in training
    text = f"<s>[INST] {sample['instruction']} [/INST] {sample['output']} </s>"
    
    # Tokenize
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    
    # Forward pass
    outputs = model(**inputs, labels=inputs['input_ids'])
    loss = outputs.loss
    
    # Backward pass (compute gradients)
    model.zero_grad()
    loss.backward()
    
    # Accumulate squared gradients (Fisher Information)
    for name, param in model.named_parameters():
        if param.requires_grad:
            if param.grad is not None:
                fisher_dict[name] += param.grad.data ** 2
            else:
                # Safe fallback for unused params
                fisher_dict[name] += torch.zeros_like(param)

# Normalize by N samples
for name in fisher_dict:
    fisher_dict[name] /= SAMPLES_FOR_FISHER

# Save the Fisher Matrix and Optimal Weights
torch.save({
    'fisher': fisher_dict,
    'opt_params': opt_param_dict
}, "fisher_task1.pt")
print("✅ Fisher Matrix saved to 'fisher_task1.pt'")

# 2. Create Replay Buffer
# Save raw text examples to mix into Task 2 later
print("Creating Replay Buffer...")

# Since we loaded dataset with 'load_dataset', convert a subset back to list of dicts
indices = random.sample(range(len(dataset)), k=min(REPLAY_BUFFER_SIZE, len(dataset)))
replay_data = [dataset[i] for i in indices]

with open("replay_buffer_task1.json", "w") as f:
    json.dump(replay_data, f, indent=4)
    
print(f"✅ Replay Buffer saved ({len(replay_data)} samples) to 'replay_buffer_task1.json'")

Calculating Fisher Matrix on 500 samples...


100%|██████████| 155/155 [03:28<00:00,  1.35s/it]


✅ Fisher Matrix saved to 'fisher_task1.pt'
Creating Replay Buffer...
✅ Replay Buffer saved (155 samples) to 'replay_buffer_task1.json'


In [4]:
import csv
import json
import pandas as pd

# --- Configuration ---
# Make sure this matches the file name you uploaded
QA_FILE = "/kaggle/input/philospher-stone-prashans/Harry_porter_book1_qa_pairs.txt" 
OUTPUT_CSV = "task1_evaluation_results.csv"

print(f"Phase 3: Starting Evaluation on {QA_FILE}...")

# 1. Load the Q/A Data
with open(QA_FILE, "r") as f:
    try:
        qa_data = json.load(f)
        print(f"Loaded {len(qa_data)} questions.")
    except json.JSONDecodeError:
        print("❌ Error: JSON file is invalid. Checking if it's raw text...")
        f.seek(0)
        content = f.read()
        # Attempt to parse if it's formatted differently
        try: 
             qa_data = json.loads(content)
        except:
             print("Could not parse file. Please ensure it is valid JSON format [{\"question\":...}, ...]")
             qa_data = []

# 2. Evaluation Loop
results = []
model.eval()

print("Generating responses...")
for entry in tqdm(qa_data):
    question = entry["question"]
    expected_answer = entry["answer"]
    
    # Format Prompt
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate Answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,   # Allow full, long responses
            do_sample=True,       # Creativity enabled
            temperature=0.6,      # Balanced creativity
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the model's new text
    if "[/INST]" in full_text:
        model_response = full_text.split("[/INST]")[-1].strip()
    else:
        model_response = full_text
        
    results.append([question, expected_answer, model_response])

# 3. Save to CSV
df = pd.DataFrame(results, columns=["Question", "Expected Answer", "Model Generated Answer"])
df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Evaluation Complete! Results saved to {OUTPUT_CSV}")

# Display first few rows to check
print("\n--- Preview of Results ---")
print(df.head())

Phase 3: Starting Evaluation on /kaggle/input/philospher-stone-prashans/Harry_porter_book1_qa_pairs.txt...
Loaded 20 questions.
Generating responses...


100%|██████████| 20/20 [05:44<00:00, 17.25s/it]

✅ Evaluation Complete! Results saved to task1_evaluation_results.csv

--- Preview of Results ---
                                            Question  \
0                        Where do the Dursleys live?   
1         What company does Vernon Dursley work for?   
2  Why do the Dursleys fear being associated with...   
3  What unusual behavior of animals is noticed on...   
4  What strange sight does Vernon Dursley see on ...   

                                     Expected Answer  \
0  They live at number four, Privet Drive, in Lit...   
1  He is the director of Grunnings, a company tha...   
2  They believe the Potters are strange and invol...   
3           Owls are seen flying during the daytime.   
4    He sees a cat that appears to be reading a map.   

                              Model Generated Answer  
0  The Dursleys live at number four, Privet Drive...  
1  Vernon Dursley works for Grunnings, a large an...  
2  The Dursleys were afraid of being associated w...  
3  On the